# Bodo OCR — TrOCR Fine-tuning (v2)
**Changes from v1:**
- Full dataset used (5000 train / all val / all test)
- Stage checkpoints deleted immediately after each stage to save storage
- Repetition fix: `no_repeat_ngram_size`, `repetition_penalty`, `length_penalty` added to generate calls
- `resize_token_embeddings` added so the decoder vocab matches the Bodo tokenizer
- `num_beams=4` with `early_stopping=True` for better beam search
- `decoder_start_token_id` guard against None (falls back to `bos` or 0)
- Cache disabled in OCRDataset to avoid RAM OOM with 5k samples
- `save_total_limit=1` kept — only latest checkpoint kept per stage

In [1]:
!pip install -q transformers datasets sentencepiece accelerate

In [2]:
import os
import json
import shutil
import torch
import gc
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    PreTrainedTokenizerFast
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

def free_memory(*objects):
    """Delete objects, collect garbage, empty CUDA cache."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

def disk_usage_gb(path="/kaggle/working"):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            try:
                total += os.path.getsize(fp)
            except:
                pass
    return total / 1e9

print("Ready")

Ready


In [3]:
# ── Paths ────────────────────────────────────────────────────────────────────
TOKENIZER_PATH  = "/kaggle/input/datasets/nlpimage/tokenizer/bodo_tokenizer2"
BODO_DATA_PATH  = "/kaggle/input/datasets/nlpimage/bodoordataset/Dataset"
TEST_IMAGE_PATH = f"{BODO_DATA_PATH}/photos/Bodosa1.png"

# ── Load Bodo tokenizer ───────────────────────────────────────────────────────
bodo_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_PATH)
print(f"Tokenizer vocab size: {len(bodo_tokenizer)}")

# ── Processor (ViT image processor + Bodo tokenizer) ─────────────────────────
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed",
    tokenizer=bodo_tokenizer
)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# ── Base model ────────────────────────────────────────────────────────────────
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")

# FIX: resize decoder embeddings to match our custom tokenizer vocab size.
# Without this the output projection has the wrong shape for 5000-token vocab.
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model = model.cuda()
model.gradient_checkpointing_enable()

# FIX: guard against None bos_token_id (falls back to eos, then 0)
bos_id = processor.tokenizer.bos_token_id \
       or processor.tokenizer.eos_token_id \
       or 0

model.config.decoder_start_token_id = bos_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id

# FIX: set generation config to stop repetition loops
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.repetition_penalty   = 1.5
model.generation_config.length_penalty       = 1.0

print("Base model and processor ready")

Tokenizer vocab size: 5000


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Base model and processor ready


In [4]:
class OCRDataset(Dataset):
    def __init__(self, image_dir, annotation_file, processor, max_length=64):   # changed to 64
        self.image_dir  = image_dir
        self.processor  = processor
        self.max_length = max_length
        with open(annotation_file, "r", encoding="utf-8") as f:
            self.data = json.load(f)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item  = self.data[idx]
        image = Image.open(
            os.path.join(self.image_dir, item["image"])
        ).convert("RGB")
        image.thumbnail((384, 384), Image.Resampling.LANCZOS)

        pixel_values = self.processor(
            image, return_tensors="pt"
        ).pixel_values.squeeze(0).half()

        text   = item["text"].strip()
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

In [5]:
# ── Full dataset (3500 train, all val, all test) ───────────────────────────────
train_dataset = OCRDataset(
    BODO_DATA_PATH,
    f"{BODO_DATA_PATH}/annotations/train.json",
    processor
)
# Subset to 3500
if len(train_dataset) > 3500:
    train_dataset.data = train_dataset.data[:3500]
print(f"Train: {len(train_dataset)} (subset to 3500)")

val_dataset = OCRDataset(
    BODO_DATA_PATH,
    f"{BODO_DATA_PATH}/annotations/val.json",
    processor
)
test_dataset = OCRDataset(
    BODO_DATA_PATH,
    f"{BODO_DATA_PATH}/annotations/test.json",
    processor
)
print(f"Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 3500 (subset to 3500)
Val: 2826, Test: 2830


In [6]:
class DataCollator:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        labels       = torch.stack([item["labels"]       for item in batch])
        return {"pixel_values": pixel_values, "labels": labels}

data_collator = DataCollator(processor)

In [7]:
def make_training_args(output_dir, num_epochs, lr=5e-5):
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=num_epochs,
        learning_rate=lr,
        warmup_steps=200,               # bumped: more data now
        lr_scheduler_type="cosine",
        save_strategy="epoch",
        save_total_limit=1,             # only keep latest checkpoint
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        eval_strategy="epoch",
        logging_steps=100,
        fp16=True,
        fp16_full_eval=True,
        gradient_checkpointing=True,
        report_to="none",
        resume_from_checkpoint=False,
    )

def quick_predict(mdl, proc, image_path):
    """Run a quick prediction to check model progress."""
    mdl.eval()
    image = Image.open(image_path).convert("RGB")
    pv = proc(image, return_tensors="pt").pixel_values.cuda()
    with torch.no_grad():
        ids = mdl.generate(
            pv,
            max_length=64,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            length_penalty=1.0,
        )
    return proc.batch_decode(ids, skip_special_tokens=True)[0]

def delete_stage(checkpoint_dir, model_dir):
    """Delete stage artifacts immediately to free storage."""
    for d in [checkpoint_dir, model_dir]:
        shutil.rmtree(d, ignore_errors=True)
    print(f"Deleted: {checkpoint_dir}, {model_dir}")
    print(f"Storage used: {disk_usage_gb():.2f} GB / 20 GB")

## Stage 1 — 5 epochs (warm up)

In [8]:
gc.collect()
torch.cuda.empty_cache()

trainer1 = Seq2SeqTrainer(
    model=model,
    args=make_training_args("/kaggle/working/stage1", num_epochs=5),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)
trainer1.train()

# Save stage 1 model
model.save_pretrained("/kaggle/working/model_stage1")
processor.save_pretrained("/kaggle/working/model_stage1")
print(f"Stage 1 done | Storage: {disk_usage_gb():.2f} GB")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,28.555432,13.520163
2,25.738784,12.529829
3,23.107422,11.519444
4,19.489421,10.800906
5,16.193770,10.748016


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Stage 1 done | Storage: 4.60 GB


In [9]:
# Quick sanity check after Stage 1
print("Stage 1 prediction:", quick_predict(model, processor, TEST_IMAGE_PATH))

Stage 1 prediction: सा 1 ' : ' ' ' थ ' ' स ' ' सान ल , ' आ ) आरो - ' ' ः ' ' बर ' ' 1 ' ' 3 ' 1 थि ' 1 1 ' 1 0 1 1 1 : 1 1 नि 1 1 । 1 1 सि 1 1 न 1 1 क 1 1


## Stage 2 — 10 epochs (main training)

In [10]:
# Reload from stage 1 checkpoint to get best weights
model = VisionEncoderDecoderModel.from_pretrained("/kaggle/working/model_stage1")
model.decoder.resize_token_embeddings(len(processor.tokenizer))
model = model.cuda()
model.gradient_checkpointing_enable()
model.config.decoder_start_token_id = bos_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.repetition_penalty   = 1.5

# ── Delete stage 1 artifacts NOW to free storage ─────────────────────────────
delete_stage("/kaggle/working/stage1", "/kaggle/working/model_stage1")

trainer2 = Seq2SeqTrainer(
    model=model,
    args=make_training_args("/kaggle/working/stage2", num_epochs=10),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)
trainer2.train()

model.save_pretrained("/kaggle/working/model_stage2")
processor.save_pretrained("/kaggle/working/model_stage2")
print(f"Stage 2 done | Storage: {disk_usage_gb():.2f} GB")

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Deleted: /kaggle/working/stage1, /kaggle/working/model_stage1
Storage used: 0.00 GB / 20 GB


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,17.280388,11.226513
2,15.909323,10.511663
3,12.880891,9.891190
4,9.706802,9.495200
5,7.091218,9.214287
6,5.093614,9.113711
7,3.850938,9.145277
8,2.596375,9.043228
9,2.230797,9.091544
10,2.104017,9.116199


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Stage 2 done | Storage: 4.60 GB


In [11]:
print("Stage 2 prediction:", quick_predict(model, processor, TEST_IMAGE_PATH))

Stage 2 prediction: हरि ं अ एन डे बर ' ए बर ’ बर ' थुम ' न ' गिरि त बर ' लि बर ' आ बर ' बर्ड बर बर ' 1 बर ' के बर ' बर ' हामब्लाय बर ' मि बर ' गिरि बर ' न्द बर ' थाम बर ' जारि बर बर बर थि बर ' टि बर '


## Stage 3 — 10 epochs (final fine-tuning, lower LR)

In [12]:
# Reload from stage 2 checkpoint
model = VisionEncoderDecoderModel.from_pretrained("/kaggle/working/model_stage2")
model.decoder.resize_token_embeddings(len(processor.tokenizer))
model = model.cuda()
model.gradient_checkpointing_enable()
model.config.decoder_start_token_id = bos_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.repetition_penalty   = 1.5

# ── Delete stage 2 artifacts NOW to free storage ─────────────────────────────
delete_stage("/kaggle/working/stage2", "/kaggle/working/model_stage2")

trainer3 = Seq2SeqTrainer(
    model=model,
    args=make_training_args("/kaggle/working/stage3", num_epochs=10, lr=1e-5),  # lower LR
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)
trainer3.train()
print(f"Stage 3 done | Storage: {disk_usage_gb():.2f} GB")

# Save final model and clean up stage3 checkpoint dir
model.save_pretrained("/kaggle/working/final_model")
processor.save_pretrained("/kaggle/working/final_model")
shutil.rmtree("/kaggle/working/stage3", ignore_errors=True)
print(f"Final model saved | Storage: {disk_usage_gb():.2f} GB")

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Deleted: /kaggle/working/stage2, /kaggle/working/model_stage2
Storage used: 0.00 GB / 20 GB


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,2.192837,9.186642
2,1.678670,9.246592
3,1.792838,9.225420
4,1.642585,9.256748
5,1.475240,9.394032
6,1.350829,9.354774
7,1.381945,9.316175
8,1.109050,9.344013
9,1.230375,9.347077
10,1.342977,9.357518


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


Stage 3 done | Storage: 3.45 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved | Storage: 1.15 GB


## Final Evaluation

In [13]:
# Load final model fresh for inference
final_model     = VisionEncoderDecoderModel.from_pretrained("/kaggle/working/final_model")
final_processor = TrOCRProcessor.from_pretrained("/kaggle/working/final_model")
final_model     = final_model.cuda().eval()
final_model.config.decoder_start_token_id = bos_id
final_model.config.pad_token_id           = final_processor.tokenizer.pad_token_id
final_model.config.eos_token_id           = final_processor.tokenizer.eos_token_id

print("Final model prediction:", quick_predict(final_model, final_processor, TEST_IMAGE_PATH))

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Final model prediction: रिपर्ट ं ए ए ए ' ए बर ' ए त त त गिरि ः जारि त आ लि न बर्ड अ ' ए ड त त थुम त त मि त त रे त त 1 त त बोसोरनि त त ा त त थि त त बर ' का बर ' त त राइ त त के त त ं त


In [14]:
# ── Test set evaluation (Character Error Rate) ────────────────────────────────
def cer(ref, hyp):
    """Simple character error rate."""
    import editdistance
    return editdistance.eval(ref, hyp) / max(len(ref), 1)

try:
    import editdistance
except ImportError:
    os.system("pip install -q editdistance")
    import editdistance

final_model.eval()
scores = []
for i in range(min(100, len(test_dataset))):  # sample 100 from test set
    item = test_dataset.data[i]
    ref  = item["text"].strip()
    image_path = os.path.join(BODO_DATA_PATH, item["image"])
    pred = quick_predict(final_model, final_processor, image_path)
    scores.append(cer(ref, pred))

print(f"Mean CER on 100 test samples: {sum(scores)/len(scores):.4f}")
print(f"(Lower is better; 0.0 = perfect, 1.0 = totally wrong)")

Mean CER on 100 test samples: 7.0168
(Lower is better; 0.0 = perfect, 1.0 = totally wrong)
